In [2]:
import pandas as pd
final_mut_tsv = pd.DataFrame(columns=["sample", "placements", "optimizedBlengths", "mutations"])
final_mut_tsv.shape

(0, 4)

In [ ]:
path = "/nfs/research/goldman/anoufa/pipeline/run2viridian_dir.tsv.xz"

# open path and 

In [8]:
a = set()
b = set()
c= set()
d= set()

a.add(1)
a.add(2)
b.add(2)
b.add(3)
c.add(1)
c.add(2)
d.add(3)
d.add(4)

h1 = hash((frozenset(a), frozenset(b)))
h2 = hash((frozenset(c), frozenset(d)))

h1 == h2

False

In [2]:
l

[{1, 2}, {2, 3}]

In [ ]:
test_dict = {'A': 50, 'C': 30, 'G': 20, 'T': 10}
max(test_dict, key=test_dict.get)

'A'

In [8]:
_covars = [0.000625, 0.000625, 0.000625]
_covars[:2]

[0.000625, 0.000625]

In [ ]:
from pathlib import Path

path = Path('/nfs/research/goldman/anoufa/src/dpca/6_final_filtering_and_figs.py')

Path(str(path).replace("6_final_filtering_and_figs.py", "7_find_contaminants_candidates.py"))

PosixPath('/nfs/research/goldman/anoufa/src/dpca/7_find_contaminants_candidates.py')

In [ ]:
def parse_masked(lines):
    """Return list of masked intervals (start, end)."""
    intervals = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        base, pos = parts[0], int(parts[1])
        if len(parts) == 3:
            length = int(parts[2])
        else:
            length = 1
        if base == "n":
            intervals.append((pos, pos + length - 1))
    return sorted(intervals)

def parse_mutations(lines):
    """Return list of (base, pos) or (base, pos, length)."""
    entries = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        base, pos = parts[0], int(parts[1])
        if len(parts) == 2:
            entries.append((base, pos))
    return entries

def parse_unmasked(lines):
    """Return list of (base, pos) or (base, pos, length)."""
    entries = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        base, pos = parts[0], int(parts[1])
        if len(parts) == 3:
            length = int(parts[2])
            entries.append((base, pos, length))
        else:
            entries.append((base, pos))
    return entries

def build_combined(intervals, masked_mut):

    genome_length = 29903

    results = []

    # --- 1. Mutations inside masked intervals
    for entry in masked_mut:
        if len(entry) == 3:
            base, pos, length = entry
            end = pos + length - 1
        else:
            base, pos = entry
            length = None
            end = pos
        for start_m, end_m in intervals:
            if not (end < start_m or pos > end_m):  # overlap check
                results.append(entry)
                break

    # --- 2. Complement intervals
    prev_end = 1
    for start, end in intervals:
        if start > prev_end:
            results.append(("n", prev_end, start - (prev_end + 1)))  # start of gap until start of interval
        prev_end = end + 1
    if prev_end < genome_length:
        results.append(("n", prev_end, genome_length - (prev_end)))

    return results


def parse_intervals(lines):
    """
    Parse lines into intervals (start, end) for base == 'n'.
    """
    intervals = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        base, pos = parts[0], int(parts[1])
        if len(parts) == 3:
            length = int(parts[2])
        else:
            length = 1
        if base == "N":
            intervals.append((pos, pos + length - 1))
    return sorted(intervals)

def subtract_intervals(masked_intervals, unmasked_intervals):
    """
    Return parts of masked_intervals that do not overlap with unmasked_intervals.
    """
    results = []
    for m_start, m_end in masked_intervals:
        start = m_start
        for u_start, u_end in unmasked_intervals:
            # no overlap
            if u_end < start or u_start > m_end:
                continue
            # overlap -> cut out overlap
            if u_start > start:
                results.append((start, u_start - 1))
            start = max(start, u_end + 1)
            if start > m_end:
                break
        if start <= m_end:
            results.append((start, m_end))
    return results


def generalized_hamming(seq1, seq2):
    
    IUPAC = {
    "A": {"A"}, "C": {"C"}, "G": {"G"}, "T": {"T"},
    "R": {"A","G"}, "Y": {"C","T"}, "S": {"G","C"}, "W": {"A","T"},
    "K": {"G","T"}, "M": {"A","C"},
    "B": {"C","G","T"}, "D": {"A","G","T"}, "H": {"A","C","T"}, "V": {"A","C","G"},
    "N": {"A","C","G","T","-"},
    "-": {"-"}
    }

    matches = 0
    mismatches = 0
    i=0
    for a, b in zip(seq1, seq2):
        i+=1
        if IUPAC[a.upper()] & IUPAC[b.upper()]:  # overlap
            matches += 1
        else:
            mismatches += 1
    return mismatches

In [78]:

test_unm_entry = [
    "T 2470",
    "N 2569 66",
    "N 2671 37",
    "N 2709 44",
    "N 2758 89",
    "- 3140 5",
    "T 3276",
    "A 4642",
    "N 5008 9",
    "N 5107 7",
    "N 5124 4",
    "N 5255 11",
    "G 5386",
    "N 6513 3",
    "N 7067 256",
    "A 8393",
    "A 10449",
    "N 11283 9",
    "G 11537",
    "C 13195",
    "T 15240",
    "N 16200 245",
    "G 18163",
    "N 19291 122",
    "N 19414 157",
    "N 20315 35",
    "N 20359 122",
    "T 21110",
    "N 21152 298",
    "T 21595",
    "C 21618",
    "T 21762",
    "N 21765 6",
    "T 21846",
    "N 21988 8",
    "N 22014 12",
    "N 22035 160",
    "N 22196 1",
    "N 22199 3",
    "N 22203 1",
    "N 22205 338",
    "N 22637 15",
    "K 22813",
    "N 22878 4",
    "N 22883 2",
    "A 22898",
    "T 22917",
    "A 22992",
    "C 23013",
    "G 23040",
    "A 23048",
    "G 23055",
    "C 23075",
    "A 23202",
    "T 23525",
    "G 23599",
    "A 23854",
    "A 24130",
    "T 24424",
    "A 24469",
    "T 24503",
    "T 25000",
    "T 25584",
    "T 26270",
    "G 26577",
    "A 26709",
    "C 27259",
    "N 27528 567",
    "T 28311",
    "N 28362 9",
    "N 28773 221",
    "N 29373 52",
    "G 29742"
]

masked_entry = [
    "N 1 54",
    "N 1313 283",
    "N 1958 10",
    "N 1970 66",
    "N 2045 1",
    "N 2055 810",
    "N 3121 3",
    "N 3132 35",
    "T 3276",
    "N 3508 288",
    "A 4642",
    "N 4996 307",
    "G 5386",
    "N 5947 35",
    "N 6148 28",
    "N 6513 3",
    "N 6847 486",
    "A 8393",
    "N 8613 7",
    "N 9246 17",
    "N 9266 1",
    "N 9835 172",
    "N 10011 3",
    "N 10020 80",
    "A 10449",
    "N 11283 9",
    "G 11537",
    "C 13195",
    "N 14899 295",
    "T 15240",
    "N 16187 368",
    "N 16586 174",
    "G 18163",
    "N 19276 310",
    "N 20196 301",
    "T 21110",
    "N 21147 536",
    "T 21762",
    "N 21765 6",
    "T 21846",
    "N 21987 9",
    "N 22014 1118",
    "A 23202",
    "T 23525",
    "G 23599",
    "A 23854",
    "A 24130",
    "T 24424",
    "A 24469",
    "T 24503",
    "T 25000",
    "N 25053 236",
    "T 25584",
    "T 26270",
    "N 26291 252",
    "G 26577",
    "A 26709",
    "C 27259",
    "N 27512 593",
    "T 28311",
    "N 28362 9",
    "N 28757 251",
    "N 29357 547",
]

cont_entry = [
    "T 3276",
    "A 4642",
    "N 5008 9",
    "N 5107 7",
    "N 5124 4",
    "N 5255 11",
    "G 5386",
    "N 6513 3",
    "N 7067 256",
    "A 8393",
    "A 10449",
    "N 11283 9",
    "G 11537",
    "C 13195",
    "T 15240",
    "N 16200 245",
    "G 18163",
    "N 19291 122",
    "N 19414 157",
    "N 20315 35",
    "N 20359 122",
    "T 21110",
    "N 21152 298",
    "T 21595",
    "C 21618",
    "T 21762",
    "N 21765 6",
    "T 21846",
    "N 21988 8",
    "N 22014 12",
    "N 22035 160",
    "N 22196 1",
    "N 22199 3",
    "N 22203 1",
    "N 22205 338",
    "C 23013",
    "G 23040",
    "A 23048",
    "G 23055",
    "C 23075",
    "A 23202",
    "T 23525",
    "G 23599",
    "A 23854",
    "A 24130",
    "T 24424",
    "A 24469",
    "T 24503",
    "T 25000",
    "T 25584",
    "T 26270",
    "G 26577",
    "A 26709",
    "C 27259",
    "N 27528 567",
    "T 28311",
    "N 28362 9",
    "N 28773 221",
    "N 29373 52",
    "G 29742"
]

In [65]:
masked_intervals = parse_intervals(masked_entry)
unmasked_intervals = parse_intervals(test_unm_entry)
subtracted = subtract_intervals(masked_intervals, unmasked_intervals)

print("Masked intervals:", masked_intervals)
print("Unmasked intervals:", unmasked_intervals)
print("Subtracted intervals:", subtracted)

Masked intervals: [(1, 54), (1313, 1595), (1958, 1967), (1970, 2035), (2045, 2045), (2055, 2864), (3121, 3123), (3132, 3166), (3508, 3795), (4996, 5302), (5947, 5981), (6148, 6175), (6513, 6515), (6847, 7332), (8613, 8619), (9246, 9262), (9266, 9266), (9835, 10006), (10011, 10013), (10020, 10099), (11283, 11291), (14899, 15193), (16187, 16554), (16586, 16759), (19276, 19585), (20196, 20496), (21147, 21682), (21765, 21770), (21987, 21995), (22014, 23131), (25053, 25288), (26291, 26542), (27512, 28104), (28362, 28370), (28757, 29007), (29357, 29903)]
Unmasked intervals: [(2569, 2634), (2671, 2707), (2709, 2752), (2758, 2846), (5008, 5016), (5107, 5113), (5124, 5127), (5255, 5265), (6513, 6515), (7067, 7322), (11283, 11291), (16200, 16444), (19291, 19412), (19414, 19570), (20315, 20349), (20359, 20480), (21152, 21449), (21765, 21770), (21988, 21995), (22014, 22025), (22035, 22194), (22196, 22196), (22199, 22201), (22203, 22203), (22205, 22542), (22637, 22651), (22878, 22881), (22883, 2288

In [79]:
path_ref_seq = "/nfs/research/goldman/anoufa/data/maple_ref_lower.fasta"


# GET REF SEQUENCE
with open(path_ref_seq, "r") as f:
    f.readline()  # Skip header
    ref_seq = f.readline().strip().upper()

ref_seq = list(ref_seq)
# GET VIRIDIAN UNMASKED SEQ
viridian_seq = ref_seq.copy()
for mut in parse_unmasked(test_unm_entry):
    if len(mut) == 3:
        base, pos, length = mut
        for i in range(length):
            viridian_seq[pos - 1 + i] = base
    else:
        base, pos = mut
        viridian_seq[pos - 1] = base

aux_viridian_seq = []
for start, end in subtracted:
    # ONLY KEEP POSITIONS IN THE MASKED REGIONS
    aux_viridian_seq.append("".join(viridian_seq[start - 1:end]))
aux_viridian_seq = "".join(aux_viridian_seq)

best_contaminant_score = 0
best_contaminant = ""
# CHECK THAT THE CONTAMINANT MATCHES THE VIRIDIAN CONSENSUS IN ALL THE REGIONS MASKED BY THE ALGORITHM
# rebuild the sequence only at the positions in masked_regions_alg
cont_seq = ref_seq.copy()
for mut in parse_unmasked(cont_entry):
    if len(mut) == 3:
        base, pos, length = mut
        for i in range(length):
            cont_seq[pos - 1 + i] = base
    else:
        base, pos = mut
        cont_seq[pos - 1] = base
aux_cont_seq = []
for start, end in subtracted:
    aux_cont_seq.append("".join(cont_seq[start - 1:end]))
aux_cont_seq = "".join(aux_cont_seq)

# METRIC OF HOW BOTH SEQUENCES MATCH
curr_cont_score = generalized_hamming(aux_viridian_seq, aux_cont_seq)
if curr_cont_score > best_contaminant_score:
    best_contaminant_score = curr_cont_score


No match: T vs C at position 830
No match: - vs C at position 1000
No match: - vs C at position 1001
No match: - vs T at position 1002
No match: - vs T at position 1003
No match: - vs T at position 1004
No match: A vs G at position 3518
No match: T vs G at position 3537
No match: A vs G at position 3612


In [2]:
params_path = "/nfs/research/goldman/anoufa/src/dpca/_params.py"
with open(params_path, "a") as f:
    f.write(f'\nprocessed_placement_masked = "/nfs/research/goldman/anoufa/data/MAPLE_output/processed_placements/processed_placements_results_masked_1_0.1_1_0_30000.4.tsv"')
    f.write(f'\nprocessed_placement_random = "/nfs/research/goldman/anoufa/data/MAPLE_output/processed_placements/processed_placements_results_random_1_0.1_1_0_30000.3.tsv"')

In [62]:
aux_viridian_seq[825:835]

'AAGCTCCAAA'

In [46]:
aux_cont_seq

'aaaaaaaaaaaaaaaaaaaaaaaaaataaataaaaaagaacatcgatctcttgtactaaagaaggtgccactacttgtggttacttaccccaaaatgctgttgttaaaatttattgtccagcatgtcacaattcagaagtaggacctgagcatagtcttgccgaataccataatgaatctggcttgaaaaccattcttcgtaagggtggtcgcactattgcctttggaggctgtgtgttctcttatgttggttgccataacaagtgtgcctattgggttccacgtgctagcgctaacataggttgtaaccatacaggtgttgttggagaaggttccggccgctataaatactagatggaatttcacagtattcactgagactcattgatgctatgatgttcacatctgatttgataatggcctacattacaggtggtgttgttcagttgacttcgcagtggctaactaacatctttggcactgtttatgaaaaactcaaacccgtccttgattggcttgaagagaagtttaaggaaggtgtagagtttcttagagacggttgggaaattgttaaatttatctcaacctgtgcttgtgaaattgtcggtggacaaattgtcacctgtgcaaaggaaattaaggagagtgttcagacattctttaagcttgtaaataaatttttggctttgtgtgctgactctatcattattggtggagctaaacttaaagccttgaatttaggtgaaacatttgtcacgcactcaaagggattgtacagaaagtgtgttaaatccagagaagaaactggcctactcatgcctctaaaagccccaaaagaaattatcttcttagagggagaaacacttcccacagaagtgttaacagaggaagttgtcttgaaaactggtgatttacaaccattagaacagctcgaaatcaaagacacagaaaagtactgtgccctcgtgatcagttgaactcggtacagagaaaggtaaa

In [29]:
parse_masked(masked_entry) # masked regions

inv_entry = build_combined(parse_masked(masked_entry), parse_unmasked(test_unm_entry))

# sort inv entry by the second element of the tuple
inv_entry.sort(key=lambda x: x[1])

In [24]:
55 + 1257

1312

In [30]:
inv_entry

[('N', 55, 1257),
 ('N', 1596, 361),
 ('N', 1968, 1),
 ('N', 2036, 8),
 ('N', 2046, 8),
 ('T', 2470),
 ('N', 2569, 66),
 ('N', 2671, 37),
 ('N', 2709, 44),
 ('N', 2758, 89),
 ('N', 2865, 255),
 ('N', 3124, 7),
 ('N', 3167, 340),
 ('N', 3796, 1199),
 ('N', 5008, 9),
 ('N', 5107, 7),
 ('N', 5124, 4),
 ('N', 5255, 11),
 ('N', 5303, 643),
 ('N', 5982, 165),
 ('N', 6176, 336),
 ('N', 6513, 3),
 ('N', 6516, 330),
 ('N', 7067, 256),
 ('N', 7333, 1279),
 ('N', 8620, 625),
 ('N', 9263, 2),
 ('N', 9267, 567),
 ('N', 10007, 3),
 ('N', 10014, 5),
 ('N', 10100, 1182),
 ('N', 11283, 9),
 ('N', 11292, 3606),
 ('N', 15194, 992),
 ('N', 16200, 245),
 ('N', 16555, 30),
 ('N', 16760, 2515),
 ('N', 19291, 122),
 ('N', 19414, 157),
 ('N', 19586, 609),
 ('N', 20315, 35),
 ('N', 20359, 122),
 ('N', 20497, 649),
 ('N', 21152, 298),
 ('T', 21595),
 ('C', 21618),
 ('N', 21683, 81),
 ('N', 21765, 6),
 ('N', 21771, 215),
 ('N', 21988, 8),
 ('N', 21996, 17),
 ('N', 22014, 12),
 ('N', 22035, 160),
 ('N', 22196, 1),